# PE zero-shot: de 32×32 píxeles a una categoría

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/pe_zero_shot_didactic.ipynb)

**Método elegido: PE-Core-G14-448 congelado, 80 plantillas públicas por clase e imagen completa + reflejo horizontal.**

**Kaggle comunicado por el participante: 0.96776 = 96,776%.** Es un resultado del leaderboard, no una accuracy calculada aquí ni una garantía sobre el test privado.

Primero exploraremos imágenes y resultados guardados, sin cargar los pesos. La reproducción completa es opcional y genera únicamente el CSV de PE. El notebook experimental anterior queda como historial.

**Recorrido:** problema → elección de PE → arquitectura → imágenes y transformaciones → similitud imagen/texto → predicciones → evaluación → reproducción.

## 1. El problema: cambia la representación del objeto

Train y validación contienen imágenes CIFAR-10 de 32×32 y diez categorías. En el test aparecen representaciones diferentes: juguetes, dibujos o estampados. El soporte y el material cambian mientras que la categoría representada puede mantenerse.

Eso es un **cambio de dominio**. Mejorar en la validación de fotografías no garantiza mejorar en este test. No hemos comprobado individualmente sus etiquetas.

El profesor permite datos externos, pero excluye **todo CIFAR-10-Warehouse**. Este notebook usa los archivos de la competición y pesos/plantillas publicados; no descarga Warehouse ni reconstruye etiquetas ocultas.

## 2. Por qué elegimos PE

PE permite clasificar directamente mediante conceptos textuales, conservando su representación preentrenada en lugar de ajustar una cabeza a las fotografías de CIFAR.

| Experimento | Kaggle comunicado |
|---|---:|
| ResNet-18 bilinear | 61,093% |
| ResNet-18 AugMix | 75,796% |
| DINOv3 ConvNeXt, ajuste parcial | 86,217% |
| SigLIP Giant, cabeza entrenada | 92,019% |
| DINOv3 ViT-7B, cabeza entrenada | 92,978% |
| SigLIP Giant, zero-shot | 94,015% |
| Ensemble SigLIP/DINO | 94,844% |
| **PE-Core-G, zero-shot + reflejo** | **96,776%** |

La mejora sobre el ensemble anterior es de **1,932 puntos**, aproximadamente un **37,5% menos de errores**. Elegimos PE por ese resultado observado y por la sencillez de un único modelo. No afirmamos que supere a PE+SigLIP: no se ha comunicado el score de esa combinación.

Estas comparaciones cambian arquitectura, preentrenamiento, resolución y adaptación a la vez. No prueban que una sola de esas diferencias cause la mejora. La mejor transferencia semántica es nuestra interpretación, no una causa aislada mediante ablaciones.

## 3. Arquitectura: dos encoders, un espacio compartido

PE es un modelo **imagen-texto preentrenado**, no un clasificador CIFAR entrenado desde cero. Su aprendizaje previo alinea representaciones visuales y textuales. Nosotros reutilizamos los pesos; no repetimos ese preentrenamiento.

| Componente | Configuración PE-Core-G14-448 |
|---|---|
| Torre visual | ViT; aproximadamente 1.88B parámetros |
| Entrada | 448×448 RGB |
| Parches | 14×14 → 32×32 = **1.024 tokens** |
| Transformer visual | 50 bloques; ancho 1.536; 16 cabezas de atención; MLP 8.960 |
| Posición | Embeddings absolutos y RoPE 2D |
| Agregación visual | Attention pooling con 8 cabezas; **sin token CLS** en esta variante |
| Torre textual | Aproximadamente 0.47B parámetros; 24 bloques; ancho 1.280; 20 cabezas |
| Contexto textual | Hasta 72 tokens |
| Espacio compartido | Vectores de **1.280 dimensiones** |

Cada bloque visual combina autoatención entre parches y un MLP por token, con normalizaciones y conexiones residuales. El attention pooling agrega la secuencia en un descriptor global.

**1.024 tokens no significan 1.024 detalles nuevos:** la imagen original sigue siendo de 32×32. La ampliación cambia su representación, no recupera información perdida.

Dimensiones contrastadas con la [configuración oficial fijada](https://github.com/facebookresearch/perception_models/blob/3e352cca660658d4b5c90f42a7808b11469e4c66/core/vision_encoder/config.py).

In [ ]:
from IPython.display import display, HTML
# Esquema conceptual: no representa activaciones medidas ni una explicación causal.
def architecture_svg():
    boxes = [
      (20,25,'Imagen 32 × 32','Bilinear → 448 × 448'),
      (20,125,'1.024 parches de 14 × 14','Proyección a ancho 1.536 + posición'),
      (20,225,'50 bloques Transformer visuales','Atención + MLP + conexiones residuales'),
      (20,325,'Attention pooling + proyección','Vector visual normalizado: 1.280'),
      (410,25,'10 clases × 80 plantillas públicas','Tokenizador: contexto 72'),
      (410,125,'24 bloques Transformer textuales','Vector por texto: 1.280'),
      (410,225,'Normalizar → media → normalizar','10 prototipos de dimensión 1.280'),
      (410,325,'10 similitudes coseno por vista','Media original/reflejo → argmax')]
    svg = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 750 415" role="img" aria-label="Arquitectura PE" style="max-width:100%;background:#f8fafc"><defs><marker id="arr" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#475569"/></marker></defs>'
    for x,y,title,subtitle in boxes:
        svg += f'<rect x="{x}" y="{y}" width="310" height="65" rx="10" fill="white" stroke="#2563eb"/><text x="{x+155}" y="{y+25}" text-anchor="middle" font-family="sans-serif" font-size="13" font-weight="bold">{title}</text><text x="{x+155}" y="{y+48}" text-anchor="middle" font-family="sans-serif" font-size="11">{subtitle}</text>'
    for x in (175,565):
        for y in (90,190,290):
            svg += f'<path d="M{x},{y} L{x},{y+30}" stroke="#475569" stroke-width="2" marker-end="url(#arr)"/>'
    return svg + '<path d="M330,357 L403,357" stroke="#475569" stroke-width="2" marker-end="url(#arr)"/></svg>'
display(HTML(architecture_svg()))

## 4. Preparar Colab

Para leer resultados y mostrar imágenes no hace falta cargar PE ni disponer de GPU. Para reproducir la inferencia, selecciona A100 y activa **RUN_INFERENCE** más abajo. Los resultados originales se leerán; una reproducción usará una carpeta nueva.

Se conservan PyTorch/torchvision de Colab y se verifican hashes del código del experimento original. Si la sesión ya había importado otra versión de Transformers, reiníciala tras instalar antes de continuar.

In [ ]:
%pip -q install "transformers==4.57.3" "huggingface_hub>=0.34,<1" "timm==1.0.15" "einops==0.8.1" "ftfy==6.3.1" regex kagglehub
%pip -q install matplotlib pandas pillow

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, sys
import requests

SOURCES = {
    'semantic_vit': ('fc538b90a48c2c1ce7cd8f3f96045653478744e9', '8d3baa53881282e34cbdb21c3527c7164213f3e78c1d921f393193c011958afc'),
    'pe_semantic': ('c30fc5422f23728a42cdc18741dac7ef52a7154c', '00f6ede2beb99c5abe82e3a6813e06f1802d2dedca3b82576fb2477ce951c214'),
}
CODE_DIR = Path('/content/recvis_pe_code')
CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, (revision, expected_hash) in SOURCES.items():
    url = f'https://raw.githubusercontent.com/rivas-vega/A3_Recv/{revision}/{name}.py'
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    if hashlib.sha256(response.content).hexdigest() != expected_hash:
        raise RuntimeError(f'El archivo {name} no coincide con la revisión fijada.')
    path = CODE_DIR/f'{name}.py'
    path.write_bytes(response.content)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
import semantic_vit as sv
import pe_semantic as pe
print('Código verificado:', SOURCES)

In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/A3_Recv')
PREVIOUS_PE_RUN = 'pe_g14_448_v1'
PREVIOUS_DIR = BASE/'runs'/PREVIOUS_PE_RUN
NEW_RUN_DIR = BASE/'runs'/'pe_zero_shot_didactic_v1'
RUN_INFERENCE = False  # True: reproducir PE; False: explorar resultados anteriores.
SAMPLE_SEED = 42
print('Lectura de resultados anteriores:', PREVIOUS_DIR)
print('Reproducción opcional:', NEW_RUN_DIR)
print('GPU:', torch.cuda.get_device_name() if torch.cuda.is_available() else 'Sin GPU; modo didáctico disponible')

## 5. Cargar los datos autorizados

Configura **KAGGLE_API_TOKEN** en secretos de Colab con acceso al notebook, o utiliza el login de KaggleHub. La cuenta debe haber aceptado las reglas. No publiques credenciales.

El lector valida formas, tipos e IDs. De test lee exclusivamente imágenes e IDs. Las muestras serán pequeñas, aleatorias y reproducibles; cambia SAMPLE_SEED para ver otras.

In [ ]:
import kagglehub
from google.colab import userdata
if not os.environ.get('KAGGLE_API_TOKEN'):
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
    del token
kagglehub.login()
DATA_ROOT = Path(kagglehub.competition_download('mva-recvis-2026'))
data = sv.read_data(DATA_ROOT)
CLASSES = sv.CLASSES
for split in ('train','val','test'):
    print(split, data[split].shape, data[split].dtype)

## 6. Ocho imágenes aleatorias de train y ocho de test

En train mostramos la etiqueta proporcionada. En test mostramos únicamente el ID. No anotamos el test a partir de su apariencia.

La opción nearest de Matplotlib permite ver los píxeles originales; solo controla su visualización, no el procesamiento enviado a PE.

In [ ]:
rng = np.random.default_rng(SAMPLE_SEED)
train_idx = rng.choice(len(data['train']), size=min(8,len(data['train'])), replace=False)
test_idx = rng.choice(len(data['test']), size=min(8,len(data['test'])), replace=False)
fig,axes = plt.subplots(2,8,figsize=(16,4.5),squeeze=False)
for row,(split,indices) in enumerate((('train',train_idx),('test',test_idx))):
    for ax in axes[row]: ax.axis('off')
    for ax,i in zip(axes[row],indices):
        ax.imshow(data[split][i],interpolation='nearest')
        title = f"train: {CLASSES[int(data['train_labels'][i])]}" if split=='train' else f"test ID {int(data['test_ids'][i])}"
        ax.set_title(title,fontsize=10)
fig.suptitle('Muestras aleatorias, sin selección por facilidad o acierto')
plt.tight_layout(); plt.show()

## 7. Bilinear: adaptar la resolución

La interpolación estima valores intermedios combinando píxeles vecinos. Suaviza transiciones, pero no descubre detalles desaparecidos.

**El resize es preprocesado determinista, no un aumento aleatorio de entrenamiento.** La receta elegida escala directamente **32 → 448**. No encadena los tamaños intermedios. ×2 y ×4 se muestran solo para comparar visualmente.

Todas las columnas se dibujan con nearest después del resize PIL; así Matplotlib no añade otro suavizado. El original aparece grande en pantalla para poder observarlo, pero sigue teniendo 32×32 píxeles.

In [ ]:
def bilinear_rgb(array,size):
    return Image.fromarray(np.asarray(array,dtype=np.uint8)).convert('RGB').resize((size,size),Image.Resampling.BILINEAR)

fig,axes = plt.subplots(4,5,figsize=(15,11),squeeze=False)
for ax in axes.flat: ax.axis('off')
for row,i in enumerate(test_idx[:4]):
    original = Image.fromarray(data['test'][i]).convert('RGB')
    target = bilinear_rgb(data['test'][i],448)
    views = [original,bilinear_rgb(data['test'][i],64),bilinear_rgb(data['test'][i],128),target,ImageOps.mirror(target)]
    titles = ['Original 32×32','Bilinear ×2: 64×64','Bilinear ×4: 128×128','PE: bilinear ×14, 448×448','PE: reflejo horizontal']
    for ax,im,title in zip(axes[row],views,titles):
        ax.imshow(im,interpolation='nearest'); ax.set_title(title,fontsize=10)
    axes[row,0].set_title(f"ID {int(data['test_ids'][i])} | 32×32",fontsize=10)
fig.suptitle('Mismos píxeles de origen; distintas representaciones',fontsize=14)
plt.tight_layout(); plt.show()

## 8. Las dos vistas que sí se usan

La transformación adicional del CSV elegido es el **reflejo horizontal**. Ejecutamos PE sobre la imagen completa y su reflejo y promediamos las diez puntuaciones. Los recortes del otro experimento no forman parte de este resultado.

La normalización oficial es $x_{norm}=(x/255-0.5)/0.5$ por canal: 0 pasa a −1 y 255 a +1. Contrastamos nuestra ilustración con el transform oficial, sin cargar pesos.

In [ ]:
pe.prepare_pe_source(Path('/content/recvis_pe_source'))
from core.vision_encoder.transforms import get_image_transform
reference_transform = get_image_transform(448,center_crop=False)
example = Image.fromarray(data['test'][int(test_idx[0])])
x = reference_transform(example)
rgb = np.asarray(bilinear_rgb(np.asarray(example),448),dtype=np.float32)/255
manual = (torch.from_numpy(rgb.copy()).permute(2,0,1)-0.5)/0.5
assert torch.allclose(x,manual,atol=1e-6), 'La ilustración no coincide con el transform oficial.'
print('Tensor oficial:',tuple(x.shape),'| rango:',float(x.min()),float(x.max()))
print('Diferencia máxima ilustración/oficial:',float((x-manual).abs().max()))

## 9. Cómo se decide la clase

Insertamos cada clase $c$ en las 80 plantillas públicas, **sin modificarlas**. El encoder textual $g$ produce un vector por texto. Normalizamos, promediamos y volvemos a normalizar:

$$t_c=\operatorname{norm}\left(\frac{1}{80}\sum_{k=1}^{80}\operatorname{norm}(g(q_k(c)))\right).$$

El encoder visual produce $z_v=\operatorname{norm}(f(v(x)))$ para cada vista. Calculamos:

$$s_c(x)=\frac{\langle z_{original},t_c\rangle+\langle z_{reflejo},t_c\rangle}{2},\qquad \hat y(x)=\arg\max_c s_c(x).$$

Con vectores normalizados, el producto escalar es la similitud coseno. Las diez puntuaciones corresponden a los diez conceptos textuales; no hemos entrenado una cabeza de diez clases.

**Zero-shot** significa que no ajustamos el encoder ni una cabeza con nuestras etiquetas. No significa que PE nunca haya recibido supervisión: su preentrenamiento utiliza información imagen-texto.

In [ ]:
PROMPT_DIR = Path('/content/recvis_pe_didactic_prompts')
templates,prompt_provenance = sv.load_prompt_bank(PROMPT_DIR)
print(len(templates),'plantillas ×',len(CLASSES),'clases =',len(templates)*len(CLASSES),'textos')
display(pd.DataFrame({'Plantilla pública (primeras ocho)':templates[:8]}))
print('Orden de clases:',list(enumerate(CLASSES)))

## 10. Qué uso hacemos de las etiquetas

| Información | Uso |
|---|---|
| Píxeles de train | Benchmark de velocidad y selección de batch, sin ajuste de pesos |
| Etiquetas de train | Mostrar ejemplos; no entrenar PE |
| 20% de validación para calibración | Elegir una temperatura positiva $T$ |
| 60% de validación para auditoría | Accuracy, pérdida, errores y confusión |
| 20% restante | Partición conservada del experimento anterior; no ajusta nada en PE |
| Test | Inferencia sobre imágenes e IDs; ninguna etiqueta oculta |

$$p_c(x)=\operatorname{softmax}(s(x)/T)_c.$$

La temperatura cambia la confianza, **no el argmax ni las etiquetas del CSV de PE**. La clasificación sigue siendo zero-shot; sus probabilidades sí están calibradas con validación. Una confianza de 0.99 no garantiza un 99% de acierto bajo cambio de dominio.

La validación limpia ya se utilizó durante el proyecto; no constituye una evaluación externa nueva. El score Kaggle se declara como comunicado por el participante.

## 11. Reproducir PE completo (opcional)

Con **RUN_INFERENCE=False**, solo elegimos la carpeta anterior para leer resultados. Con True, ejecutamos la rama PE de imagen completa + reflejo, desactivando mezcla con SigLIP, recortes y diagnóstico de corrupciones.

Usamos el código original fijado. La carga de pesos es estricta: FP32 para pesos, autocast BF16 para operaciones matriciales y caché FP16 de características. El modelo está congelado, sin gradientes; el transform oficial se ejecuta en CPU y el encoder en GPU.

Si existe el registro de la revisión de pesos original, lo reutilizamos. Una GPU o entorno diferente puede introducir pequeñas diferencias numéricas: no prometemos identidad bit a bit. El presupuesto de siete horas se empieza a contar al ejecutar la inferencia, no mientras lees.

In [ ]:
if RUN_INFERENCE:
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Para reproducir, selecciona A100 o una GPU compatible con BF16.')
    torch.manual_seed(42)
    torch.set_float32_matmul_precision('high')
    SETTINGS = dict(seed=42,previous_run=str(PREVIOUS_DIR),crop_size=28,uncertain_fraction=.25,crop_weight=.25,
                    run_local_views=False,run_stress_audit=False)
    # Los parámetros de recortes son inactivos; se conservan por compatibilidad.
    class PEZeroShotOnly(pe.PEExperiment):
        def blend_previous(self):
            print('PE solo: no se carga ningún resultado de SigLIP.')
    model_record = Path('models')/(pe.MODEL_ID.replace('/','__')+'.json')
    previous_record = PREVIOUS_DIR/model_record
    new_record = NEW_RUN_DIR/model_record
    if previous_record.exists():
        record=json.loads(previous_record.read_text())
        if record['model_id']!=pe.MODEL_ID:
            raise ValueError('El registro anterior no corresponde a PE.')
        if new_record.exists() and json.loads(new_record.read_text())!=record:
            raise ValueError('Revisión de pesos distinta: utiliza otra carpeta de reproducción.')
        sv.atomic_json(record,new_record)
    budget=sv.Budget(7.0)
    experiment=PEZeroShotOnly(data,SETTINGS,NEW_RUN_DIR,BASE/'cache_pe_semantic',
                              Path('/content/recvis_pe_chunks'),budget)
    sv.atomic_json(dict(sources=SOURCES,pe_upstream_revision=pe.PE_REVISION),NEW_RUN_DIR/'code_revision.json')
    experiment.run()
    RESULT_DIR=NEW_RUN_DIR
else:
    RESULT_DIR=PREVIOUS_DIR
    print('Modo lectura: sin inferencia ni modificación de resultados anteriores.')
print('Resultados:',RESULT_DIR)

## 12. Recuperar probabilidades y verificar su procedencia

Necesitamos **pe_zero_shot_probabilities.npz**, no solamente el CSV. Antes de relacionar predicciones e imágenes verificamos hashes de datos, orden de clases, particiones e IDs.

Si falta el archivo, las figuras anteriores siguen disponibles. Corrige PREVIOUS_PE_RUN o activa RUN_INFERENCE. No se fabrican predicciones ni métricas.

In [ ]:
P_VAL=P_TEST=None
PROB_FILE=RESULT_DIR/'pe_zero_shot_probabilities.npz'
METRIC_FILE=RESULT_DIR/'pe_zero_shot_metrics.json'
if PROB_FILE.exists():
    manifest=json.loads((RESULT_DIR/'experiment.json').read_text())
    assert manifest['data_hashes']==data['hashes'], 'Predicciones de otros datos.'
    assert manifest['classes']==list(CLASSES), 'Orden de clases diferente.'
    assert manifest['validation_parts']==[p.tolist() for p in data['parts']], 'Particiones diferentes.'
    with np.load(PROB_FILE,allow_pickle=False) as saved:
        assert np.array_equal(saved['test_ids'],data['test_ids']), 'IDs desalineados.'
        val_p,test_p=saved['val'].copy(),saved['test'].copy()
    for split,prob in [('val',val_p),('test',test_p)]:
        assert prob.shape==(len(data[split]),10)
        assert np.isfinite(prob).all() and (prob>=0).all()
        assert np.allclose(prob.sum(axis=1),1,atol=2e-5)
    P_VAL,P_TEST=val_p,test_p
    print('Probabilidades verificadas:',P_VAL.shape,P_TEST.shape)
    if METRIC_FILE.exists():
        metrics=json.loads(METRIC_FILE.read_text())
        print('Temperatura guardada:',metrics.get('temperature'))
else:
    print('No se encuentra:',PROB_FILE)
    print('Corrige PREVIOUS_PE_RUN o activa RUN_INFERENCE y ejecuta desde su celda.')

## 13. Test: ejemplos aleatorios y top-3

Mostramos las mismas imágenes elegidas antes de leer las predicciones. Las barras representan **probabilidades del modelo**, no accuracy ni etiquetas verdaderas. No utilizamos esta figura para anotar el test.

In [ ]:
if P_TEST is not None:
    fig,axes=plt.subplots(4,2,figsize=(11,11),gridspec_kw={'width_ratios':[1,2]},squeeze=False)
    for row,i in enumerate(test_idx[:4]):
        axes[row,0].imshow(bilinear_rgb(data['test'][i],448),interpolation='nearest')
        axes[row,0].axis('off');axes[row,0].set_title(f"ID {int(data['test_ids'][i])} — sin etiqueta conocida")
        best=np.argsort(P_TEST[i])[-3:][::-1]
        axes[row,1].barh(np.arange(3),P_TEST[i,best],color=['#2563eb','#94a3b8','#cbd5e1'])
        axes[row,1].set_yticks(np.arange(3),[CLASSES[j] for j in best]);axes[row,1].invert_yaxis()
        axes[row,1].set_xlim(0,1);axes[row,1].set_xlabel('Probabilidad calibrada, no certeza de acierto')
    plt.tight_layout();plt.show()
else:
    print('Esta sección necesita probabilidades verificadas.')

## 14. Dónde se equivoca: validación etiquetada

Aquí sí conocemos las etiquetas. Usamos la partición de auditoría para la matriz de confusión y una muestra aleatoria de errores. El resultado mide ese dominio, no sustituye Kaggle.

In [ ]:
if P_VAL is not None:
    audit=data['parts'][2]
    truth=data['val_labels'][audit];pred=P_VAL[audit].argmax(axis=1)
    accuracy=float(np.mean(pred==truth))
    print(f'Accuracy de validación reservada: {accuracy:.3%} ({len(audit):,} imágenes)')
    print('Kaggle comunicado: 96.776%; no calculado aquí.')
    confusion=np.zeros((10,10),dtype=np.int64)
    np.add.at(confusion,(truth,pred),1)
    normalized=confusion/np.maximum(confusion.sum(axis=1,keepdims=True),1)
    fig,ax=plt.subplots(figsize=(9,7))
    im=ax.imshow(normalized,vmin=0,vmax=1,cmap='Blues')
    ax.set_xticks(range(10),CLASSES,rotation=45,ha='right');ax.set_yticks(range(10),CLASSES)
    ax.set_xlabel('Predicción');ax.set_ylabel('Etiqueta de validación')
    ax.set_title('Confusión normalizada por clase real')
    for r in range(10):
        for c in range(10):
            ax.text(c,r,f'{normalized[r,c]:.0%}',ha='center',va='center',fontsize=8,
                    color='white' if normalized[r,c]>.5 else 'black')
    fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()
    errors=audit[pred!=truth]
    if len(errors):
        chosen=np.random.default_rng(SAMPLE_SEED).choice(errors,size=min(8,len(errors)),replace=False)
        fig,axes=plt.subplots(2,4,figsize=(12,6),squeeze=False)
        for ax in axes.flat:ax.axis('off')
        for ax,i in zip(axes.flat,chosen):
            ax.imshow(data['val'][i],interpolation='nearest')
            ax.set_title(f"Real: {CLASSES[int(data['val_labels'][i])]}\nPE: {CLASSES[int(P_VAL[i].argmax())]}",fontsize=10)
        fig.suptitle('Errores aleatorios de validación; no son imágenes del test')
        plt.tight_layout();plt.show()
else:
    print('Esta sección necesita probabilidades verificadas.')

## 15. Qué hemos aprendido y qué falta demostrar

- El mejor resultado comunicado de este recorrido es PE zero-shot con dos vistas: **96,776%**.
- No hemos aislado el efecto de 448 frente a otros tamaños ni el efecto individual del reflejo. Seguimos el preprocesado del checkpoint y el protocolo ejecutado.
- Varios prototipos textuales serían otro experimento: **no forman parte de este resultado**.
- No hemos usado superresolución generativa, entrenado con imágenes sintéticas nuevas ni recuperado etiquetas de Warehouse.
- Llegar al 98% exige reducir aproximadamente el 38% de los errores restantes respecto al 96,776%; aumentar el modelo no lo garantiza.

Para el informe declara checkpoint, preentrenamiento, plantillas, resolución, dos vistas, calibración y score. Este notebook explica y reproduce el método; no sustituye el informe de una página del curso.

## 16. Descargar únicamente el CSV elegido

Se regenera desde las probabilidades verificadas, respetando exactamente los IDs y el orden de sample_submission.csv. No se envía nada automáticamente a Kaggle.

Antes de publicar una copia, borra las salidas: pueden contener imágenes privadas de la competición. El notebook del repositorio no contiene salidas, datos ni tokens.

In [ ]:
if P_TEST is not None:
    output=Path('/content/submission_pe_zero_shot.csv')
    sv.write_submission(data['test_ids'],P_TEST,data['files']['sample_submission.csv'],output)
    from google.colab import files
    files.download(str(output))
else:
    print('Primero recupera o calcula las probabilidades de PE.')

## Referencias y reproducibilidad

- [Perception Encoder: artículo](https://arxiv.org/abs/2504.13181) y [checkpoint PE-Core-G14-448](https://huggingface.co/facebook/PE-Core-G14-448).
- [Arquitectura oficial fijada](https://github.com/facebookresearch/perception_models/blob/3e352cca660658d4b5c90f42a7808b11469e4c66/core/vision_encoder/config.py) y [transformación oficial](https://github.com/facebookresearch/perception_models/blob/3e352cca660658d4b5c90f42a7808b11469e4c66/core/vision_encoder/transforms.py).
- [80 plantillas públicas OpenCLIP](https://github.com/mlfoundations/open_clip/blob/4cc208d3a44a2fb6d6435006a46f520d2ccc29b1/src/open_clip/zero_shot_metadata.py); licencia conservada en el repositorio.
- [Código PE utilizado](https://github.com/rivas-vega/A3_Recv/blob/c30fc5422f23728a42cdc18741dac7ef52a7154c/pe_semantic.py).

Se guardan hashes de datos, revisiones, versiones, temperatura, probabilidades e IDs. El score 0.96776 fue comunicado por el participante el 12 de noviembre de 2026; no procede de consultar etiquetas ocultas.

**Verificación de este notebook:** estructura y contenido revisados; celdas nuevas de visualización pendientes de ejecución en Colab. El entorno de desarrollo se desconectó durante su preparación. Reutiliza el motor de inferencia fijado que produjo el experimento anterior.